# Ordered Logistic Regression Results for Adoption Predictors: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² ordered logistic regression results dataset using the `mlcroissant` library and Python data tools.

### Dataset Source
The dataset is described in Croissant metadata format and is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load Croissant metadata and available records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}\n")
print(f"Version: {getattr(metadata, 'version', None)}\n")


## 2. Data Overview
Explore available record sets and their fields. All entities are referenced by their `@id`.

The Croissant schema describes record sets, with each record set containing fields and/or columns (each with an `@id`).

In [ ]:
# List all available record sets (`cr:RecordSet`) by @id and name
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Record Sets available in this dataset:")
    for rs in record_sets:
        print(f"  @id: {rs['@id']} | name: {rs.get('name', 'n/a')}")

# For demo: collect field/column @ids for each RecordSet
rs_fields = {}
for rs in record_sets:
    rs_id = rs['@id']
    fields = rs.get('field', [])
    # Ensure fields is always a list
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = [(f['@id'], f.get('name', '')) for f in fields]
    rs_fields[rs_id] = field_ids

    print(f"\nFields in record set @{rs_id}:")
    for fid, fname in field_ids:
        print(f"   field @id: {fid} | name: {fname}")

## 3. Data Extraction
Load data from each available record set into DataFrames, indexed by record set `@id`.
Use field `@id`s for selecting columns. If no record set exists in the schema, inform the user.

In [ ]:
# Build a dictionary of DataFrames for each record set by @id
dataframes = {}

if not record_sets:
    print("No record sets are defined in this dataset; data extraction is not possible.")
else:
    for rs in record_sets:
        rs_id = rs['@id']
        # Extract record generator and convert to DataFrame
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record set @{rs_id} loaded: {df.shape[0]} rows, {df.shape[1]} columns")

    # Show columns for first available record set as example
    selected_id = record_sets[0]['@id']
    print(f"\nColumns for record set @{selected_id}:")
    print(dataframes[selected_id].columns.tolist())
    display(dataframes[selected_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform basic filtering and normalization of a numeric field, and group by another field (when available).

You should set the appropriate field `@id`s for your dataset—refer to the overview above.

In [ ]:
# For demonstration, select record set @id and valid field @id(s)
if not dataframes:
    print("No DataFrame has been loaded; EDA cannot proceed.")
else:
    # Choose demonstration record set (first available)
    rs_id = next(iter(dataframes))
    df = dataframes[rs_id]
    
    # Identify numeric fields
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print("No numeric fields found for EDA in the selected record set.")
    else:
        numeric_field_id = numeric_cols[0]  # Use first numeric column
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized \"{numeric_field_id}\" for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Identify categorical for grouping
        group_candidate_cols = df.select_dtypes(include='object').columns.tolist()
        if group_candidate_cols:
            group_field = group_candidate_cols[0]
            print(f"\nGrouping by @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame("mean_" + numeric_field_id)
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize a numeric field's distribution or its relationship to a categorical grouping, if available.

_Note: Depending on your runtime, plots may need to be rendered inline; ensure proper `%matplotlib inline` magic if necessary._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data to visualize.")
else:
    df = dataframes[rs_id]
    if numeric_cols:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of field @{numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant-annotated dataset using the `mlcroissant` library.

- Metadata and structure are accessed programmatically and referenced via their `@id`s.
- Record set and field IDs allow for reproducible data access and robust transformations.
- You can now extend this workflow for more detailed statistical analysis or machine learning tasks relevant to rangeland management and knowledge adoption.